# Cálculo ráster y almacenamiento de resultados en carpetas organizadas / Raster calculation & save results in organized folders

**ES:** Este cuaderno le ayudará a procesar archivos ráster (imágenes TIF) en 5 pasos sencillos. Comparamos los escenarios FLR (Restauración del Paisaje Forestal) y BAU (Escenario sin cambios) para ver las diferencias, y luego guardamos los resultados en carpetas organizadas.

## Qué hace este código (5 pasos sencillos):
1. **Encontrar sus mapas** en ArcGIS Pro
2. **Buscar capas ráster** (archivos TIF) en un grupo
3. **Emparejar archivos FLR y BAU** (como emparejar gemelos)
4. **Crear carpetas organizadas** con resultados
5. **Calcular diferencias** entre los pares

## Antes de comenzar:
- Tener archivos **tif** o **tiff** organizados en una capa de grupo
- Asegúrese de que sus archivos tengan "FLR" y "BAU" en sus nombres

## Cómo usar:
Ejecute cada paso en orden presionando **Shift + Enter** en cada celda. Cada paso se basa en el anterior.

---

**EN:** This notebook will help you process raster files (TIF images) in 5 simple steps. We compare FLR (Forest Landscape Restoration) and BAU (Business As Usual) scenarios to see the differences, and then save results in organized folders.

## What this code does (5 Simple Steps):
1. **Find your maps** in ArcGIS Pro
2. **Look for raster layers** (TIF files) in a group
3. **Match FLR and BAU pairs** (like matching twins)
4. **Create organized folders** with results
5. **Calculate differences** between the pairs

## Before you start:
- Have **tif** or **tiff** files organized in a group layer
- Make sure your files have "FLR" and "BAU" in their names

## How to use:
Execute each step in order by pressing **Shift + Enter** on each cell. Each step builds on the previous one.

In [ ]:
# STEP 1: Setup and find maps in ArcGIS Pro
print("🗺️ STEP 1: Finding your maps in ArcGIS Pro...")
print("=" * 50)

# Import all the tools we need
import arcpy          # This talks to ArcGIS
from arcpy.sa import * # This gives us raster calculation tools
import os             # This helps us work with files and folders
import re             # This helps us find patterns in text

# Cargar variables del .env (ruta portable) / Load .env (portable)
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

# Configuration
arcpy.env.overwriteOutput = True
# Get permission to use Spatial Analyst (needed for raster math)
arcpy.CheckOutExtension("Spatial")

# 📝 HELPER FUNCTIONS (we'll use these later)
def safe_input(prompt, default=None):
    """Safely get user input with default fallback"""
    try:
        val = input(prompt).strip()
        return val if val else default
    except (EOFError, KeyboardInterrupt):
        return default

def create_directory(base_path, folder_name):
    """Create a new folder if it doesn't exist yet"""
    full_path = os.path.join(base_path, folder_name)
    if not os.path.exists(full_path):
        os.makedirs(full_path)
        print(f"📁 Created new folder: {full_path}")
    return full_path

def extract_common_name(name1, name2):
    """Find common parts in file names for directory naming"""
    name1_clean = os.path.splitext(os.path.basename(name1))[0]
    name2_clean = os.path.splitext(os.path.basename(name2))[0]
    
    # Look for RCP pattern (like RCP85_50)
    rcp_pattern = r'RCP(\d+)_(\d+)'
    match1 = re.search(rcp_pattern, name1_clean)
    match2 = re.search(rcp_pattern, name2_clean)
    
    if match1 and match2 and match1.groups() == match2.groups():
        rcp_scenario = match1.group(1)
        time_period = match1.group(2)
        common_name = f"usle_RCP{rcp_scenario}_{time_period}_midterm"
        print(f"   ✅ Found RCP pattern: {common_name}")
        return common_name
    
    # If no RCP pattern, find other common words
    print("   🔍 No RCP pattern found, looking for common words...")
    words1 = re.split(r'[_\-\s]', name1_clean)
    words2 = re.split(r'[_\-\s]', name2_clean)
    common_parts = [word for word in words1 if word in words2 and word.lower() not in ['flr', 'bau']]
    
    return '_'.join(common_parts) if common_parts else 'raster_comparison'

print("✅ Helper functions ready!")

# Now let's find your maps!
print("\n🔍 Looking for maps in your ArcGIS project...")

# Connect to the current ArcGIS project
project = arcpy.mp.ArcGISProject("CURRENT")
all_maps = project.listMaps()

print(f"📊 Found {len(all_maps)} map(s) in your project:")

# Check how many maps we found and handle accordingly
if len(all_maps) == 0:
    print("❌ ERROR: No maps found in the current project.")
    print("💡 Make sure you have ArcGIS Pro open with a project loaded!")
    raise Exception("No maps found in the current project.")

elif len(all_maps) == 1:
    # Perfect! Just one map
    chosen_map = all_maps[0]
    print(f"✅ Only one map found: '{chosen_map.name}'")
    print(f"🎯 We'll use this map automatically!")
    
else:
    # Multiple maps - let user choose
    print("📋 Multiple maps available:")
    for i, map_obj in enumerate(all_maps, 1):
        print(f"   {i}. {map_obj.name}")
    
    # Ask user to select
    map_name_to_use = safe_input("👉 Type the exact name of the map you want to use: ")
    
    chosen_map = next((m for m in all_maps if m.name == map_name_to_use), None)
    
    if not chosen_map:
        print(f"❌ ERROR: Could not find a map named '{map_name_to_use}'")
        raise Exception(f"Map '{map_name_to_use}' not found.")
    else:
        print(f"✅ Selected map: '{chosen_map.name}'")

print(f"\n🎉 STEP 1 COMPLETE!")
print(f"📝 We found and selected your map: '{chosen_map.name}'")
print(f"⏭️ Ready for Step 2: Looking for raster layers!")

In [ ]:
# STEP 2: Find raster layers in group layers
print("📁 STEP 2: Looking for raster layers in group layers...")
print("=" * 50)
print("💡 Group layers are like folders that contain other layers")

# Get all layers from our chosen map
all_layers = chosen_map.listLayers()

# Filter to find only group layers (the folders)
group_layers = [lyr for lyr in all_layers if lyr.isGroupLayer]

if not group_layers:
    raise Exception("No group layers found in the map.")

print("📋 Available groups (choose one):")
for i, group in enumerate(group_layers, 1):
    layers_inside = group.listLayers()
    print(f"   {i}. 📁 {group.name} (contains {len(layers_inside)} layer(s))")

# Ask the user which group layer they want to use
group_name_to_use = safe_input("\n👉 Type the exact name of the group layer containing TIF files: ")
selected_group = next((g for g in group_layers if g.name == group_name_to_use), None)

if not selected_group:
    raise Exception(f"Group layer '{group_name_to_use}' not found.")

print(f"✅ Great! Selected group: '{selected_group.name}'")

# Find TIF files in the selected group
tif_files_found = []
for layer in selected_group.listLayers():
    if hasattr(layer, 'dataSource') and layer.dataSource:
        if layer.dataSource.lower().endswith(('.tif', '.tiff')):
            tif_files_found.append(layer)

if not tif_files_found:
    raise Exception(f"No TIF files found in group '{selected_group.name}'")

print(f"✅ Found {len(tif_files_found)} TIF file(s) in '{selected_group.name}'")

print(f"\n🎉 STEP 2 COMPLETE!")
print(f"📝 Found {len(tif_files_found)} TIF files in group '{selected_group.name}'")
print(f"⏭️ Ready for Step 3: Matching FLR and BAU pairs!")


In [ ]:
# STEP 3: Match FLR and BAU pairs
print("🔗 STEP 3: Match FLR and BAU pairs")
print("=" * 50)

# Filter TIF files to find FLR and BAU pairs
# FLR files contain 'flr' in their name, BAU files contain 'bau' in their name
flr_files = [lyr for lyr in tif_files_found if 'flr' in lyr.name.lower()]
bau_files = [lyr for lyr in tif_files_found if 'bau' in lyr.name.lower()]

if not flr_files or not bau_files:
    raise Exception("Need both FLR and BAU files to proceed.")

print(f"Found {len(flr_files)} FLR files and {len(bau_files)} BAU files")

# Find matching pairs
matching_pairs = []
for flr_file in flr_files:
    flr_name = flr_file.name.lower()
    
    # Method 1: RCP pattern matching
    rcp_pattern = r'(usle_rcp\d+_\d+)_flr'
    rcp_match = re.search(rcp_pattern, flr_name)
    
    matching_bau = None
    if rcp_match:
        base_pattern = rcp_match.group(1)
        for bau_file in bau_files:
            if base_pattern in bau_file.name.lower() and 'bau' in bau_file.name.lower():
                matching_bau = bau_file
                break
    else:
        # Method 2: Simple replacement
        expected_bau_name = flr_name.replace('flr', 'bau')
        for bau_file in bau_files:
            if bau_file.name.lower() == expected_bau_name:
                matching_bau = bau_file
                break
    
    if matching_bau:
        matching_pairs.append((flr_file, matching_bau))

if not matching_pairs:
    raise Exception("No matching FLR/BAU pairs found.")

print(f"Matched {len(matching_pairs)} FLR/BAU pairs:")
for i, (flr, bau) in enumerate(matching_pairs, 1):
    print(f"  {i}. {flr.name} ↔ {bau.name}")


In [ ]:
# STEP 4: Create organized folders with results
print("STEP 4: CREATE ORGANIZED FOLDERS WITH RESULTS")
print("=" * 50)

# Set up a default location for saving files
# La carpeta de salida se lee del .env (RASTER_OUTPUT_DIR), basada en OneDrive de IUCN.
# / Output folder comes from .env (RASTER_OUTPUT_DIR), based on IUCN OneDrive.
default_save_location = os.environ.get("RASTER_OUTPUT_DIR") or os.path.join(os.getcwd(), "RasterComparisons")

print(f"\n💾 Where would you like to save the comparison results?")
print(f"📍 Default location: {default_save_location}")
print(f"💡 Just press Enter to use the default, or type a new path")

# Ask the user where they want to save files
output_folder = safe_input(
    f"\n👉 Enter output directory path (or press Enter for default): ",
    default_save_location
)

print(f"\n📂 You chose: {output_folder}")

# Check if the folder exists, and create it if it doesn't
if not os.path.exists(output_folder):
    print(f"📁 This folder doesn't exist yet, so I'll create it...")
    try:
        os.makedirs(output_folder)
        print(f"✅ Created new folder: {output_folder}")
    except Exception as e:
        print(f"❌ ERROR: Could not create folder: {e}")
        print("💡 Make sure the path is valid and you have permission to create folders there")
        raise e
else:
    print(f"✅ Folder already exists and is ready to use!")



In [ ]:
# STEP 5: Calculate differences between pairs
print("🧮 STEP 5: Calculating differences between FLR and BAU pairs...")
print("=" * 50)
print("💡 We'll calculate: FLR - BAU = Difference")
# Keep track of our progress
success_count = 0  # Count successful processing
failure_count = 0  # Count failed processing
total_pairs = len(matching_pairs)

print(f"\n📊 We have {total_pairs} pairs to process")
print("🕐 This might take a while depending on file sizes...")

# Process each pair one by one
for pair_number, (flr_file, bau_file) in enumerate(matching_pairs, 1):
    
    print(f"\n{'='*50}")
    print(f"🔄 Processing Pair {pair_number} of {total_pairs}")
    print(f"{'='*50}")
    
    try:
        # Show which files we're working on
        print(f"🌱 FLR file: {flr_file.name}")
        print(f"🏭 BAU file: {bau_file.name}")
        
        # Create folder name
        print(f"📋 Creating output folder...")
        folder_name = extract_common_name(flr_file.name, bau_file.name)
        result_folder = create_directory(output_folder, folder_name)
        
        # Plan output file names
        difference_file = os.path.join(result_folder, f"{folder_name}_difference.tif")
        impact_file = os.path.join(result_folder, f"{folder_name}_impact.tif")
        
        print(f"   📄 Difference raster: {difference_file}")
        print(f"   📄 Impact raster: {impact_file}")
        
        # Get file paths
        flr_path = flr_file.dataSource
        bau_path = bau_file.dataSource
        
        print(f"   📁 FLR source: {flr_path}")
        print(f"   📁 BAU source: {bau_path}")
        
        # Calculate difference
        print(f"🧮 Calculating raster difference (FLR - BAU)...")
        flr_raster = Raster(flr_path)
        bau_raster = Raster(bau_path)
        difference_raster = flr_raster - bau_raster
        difference_raster.save(difference_file)
        print(f"   ✅ Difference calculation complete!")
        
        # Create impact map (remove zero values)
        print(f"🎯 Creating impact map (removing zero values)...")
        print("   💡 This highlights only areas with actual changes")
        impact_raster = SetNull(difference_raster == 0, difference_raster)
        impact_raster.save(impact_file)
        print(f"   ✅ Impact map creation complete!")
        
        # Success!
        success_count += 1
        print(f"✅ Pair {pair_number} completed successfully!")
        print(f"📁 Results saved in: {result_folder}")
        
    except Exception as error:
        # Something went wrong
        failure_count += 1
        print(f"❌ Pair {pair_number} failed!")
        print(f"💡 Error details: {error}")
        print(f"⏭️ Continuing with next pair...")

# Show processing summary
print(f"\n{'='*60}")
print(f"📊 PROCESSING SUMMARY:")
print(f"{'='*60}")
print(f"   ✅ Successfully processed: {success_count} pairs")
print(f"   ❌ Failed to process: {failure_count} pairs")

# Calculate success percentage
if total_pairs > 0:
    success_rate = (success_count / total_pairs) * 100
    print(f"   📈 Success rate: {success_rate:.1f}%")

if failure_count > 0:
    print(f"\n⚠️ Some pairs failed to process.")
    print(f"💡 Common reasons for failure:")
    print(f"   • File permission issues")
    print(f"   • Corrupted raster files")
    print(f"   • Insufficient disk space")
    print(f"   • ArcGIS licensing issues")

print(f"\n🎉 STEP 5 COMPLETE!")
print(f"📝 Processed {success_count} out of {total_pairs} raster pairs")
print(f"📁 Results saved in: {output_folder}")

In [ ]:
# Display the folder structure that was created
print(f"\n📁 Results organized in: {output_folder}")
print(f"📊 Total folders created: {success_count}")
print(f"📄 Total files created: {success_count * 2}")

print(f"\n📂 Folder structure:")
for i, (flr, bau) in enumerate(matching_pairs[:success_count], 1):
    folder_name = extract_common_name(flr.name, bau.name)
    folder_path = os.path.join(output_folder, folder_name)
    
    print(f"\n  📁 {folder_name}/")
    print(f"     📄 {folder_name}_difference.tif  (FLR - BAU)")
    print(f"     📄 {folder_name}_impact.tif      (non-zero changes only)")

print(f"\n✅ Processing workflow completed!")
print(f"📈 Success rate: {(success_count/total_pairs)*100:.1f}% ({success_count}/{total_pairs})")


# Guía de personalización / Customization Guide

**ES:** Esta guía muestra cómo modificar el flujo de trabajo para sus necesidades específicas. Concéntrese en estas áreas clave de personalización:

---

## **IMPORTANTE: Puede cambiar la carpeta de salida en el Paso 4**

El **Paso 4** le pregunta dónde guardar los resultados. Puede:
- Presionar **Enter** para usar la ubicación predeterminada
- Escribir cualquier ruta que desee (ej. `C:\MisResultados`, `D:\GIS\Output`)
- El código creará la carpeta si no existe

---

## 1. **Diferentes cálculos** (modificar Paso 5)

Si necesita cálculos diferentes en lugar de `FLR - BAU`, reemplace esta sección en el **Paso 5**:

### Código original:
```python
difference_raster = flr_raster - bau_raster
difference_raster.save(difference_file)
```

### Ejemplo 1: Cambio porcentual
```python
percent_change = ((flr_raster - bau_raster) / bau_raster) * 100
percent_change.save(difference_file)
```

### Ejemplo 2: Análisis de proporción
```python
ratio_raster = flr_raster / bau_raster
ratio_raster.save(difference_file)
```

---

## 2. **Diferentes tipos de archivo** (modificar Paso 2)

### Código actual busca archivos TIF:
```python
if layer.dataSource.lower().endswith(('.tif', '.tiff')):
    tif_files_found.append(layer)
```

### Para incluir otros formatos:
```python
supported_formats = ('.tif', '.tiff', '.img', '.ecw', '.jp2', '.grd')
if layer.dataSource.lower().endswith(supported_formats):
    tif_files_found.append(layer)
```

---

## 3. **Diferentes patrones de nombre** (modificar Paso 3)

### Código actual busca 'FLR' y 'BAU':
```python
flr_files = [lyr for lyr in tif_files_found if 'flr' in lyr.name.lower()]
bau_files = [lyr for lyr in tif_files_found if 'bau' in lyr.name.lower()]
```

### Ejemplo: Buscar 'RESTORED' y 'BASELINE':
```python
flr_files = [lyr for lyr in tif_files_found if 'restored' in lyr.name.lower()]
bau_files = [lyr for lyr in tif_files_found if 'baseline' in lyr.name.lower()]
```

---

## Resumen de personalizaciones más comunes / Most common customizations summary

| **Qué desea / What you want** | **Paso a modificar / Step to modify** | **Qué cambiar / What to change** |
|-------------------------------|--------------------------------------|--------------------------------|
| Diferente ubicación de guardado / Different save location | Paso/Step 4 | Escriba la nueva ruta cuando se le solicite / Type new path when prompted |
| Diferentes tipos de archivo / Different file types | Paso/Step 2 | Cambie la lista de extensiones / Change file extensions list |
| Diferentes palabras clave / Different keywords | Paso/Step 3 | Cambie 'flr' y 'bau' por sus términos / Change 'flr' and 'bau' to your terms |
| Diferente cálculo / Different calculation | Paso/Step 5 | Reemplace la operación matemática / Replace the math operation |

---

**EN:** This guide shows how to modify the workflow for your specific needs. Focus on the key customization areas listed above. Always test your modifications on a small subset of data first to make sure everything works.